# Multi-Session Cell Tuning Analysis - CORRECTED

This notebook analyzes tuning properties across **all sessions** in the dataset using **corrected alignment** for STOP/CONT trials.

**IMPORTANT**: STOP/CONT trials are aligned to `stop_cue` (not `go_cue`) to properly measure signal responses.

**Objectives:**
1. Run tuning analysis on all cells across all sessions
2. Generate session-level statistics
3. Compare tuning properties across sessions
4. Identify sessions with high/low proportions of tuned cells
5. Generate comprehensive visualizations
6. Save results for each session

**Key Correction:**
- STOP/CONT trials now aligned to `stop_cue` to measure actual signal responses
- New measures: STOP modulation, CONT modulation, Signal discrimination, GO vs Signals

In [ ]:
# Imports
import sys
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import holoviews as hv
import hvplot.pandas
hv.extension('bokeh')

from scipy import stats
from scipy.stats import f_oneway, ttest_ind
from itertools import combinations
from tqdm.auto import tqdm
import matplotlib.pyplot as plt
import seaborn as sns

# Add parent directory to path
sys.path.insert(0, str(Path.cwd().parent))

# Import classes
import importlib
import session_class
import cell_analysis
importlib.reload(session_class)
importlib.reload(cell_analysis)
from session_class import Session
from cell_analysis import Cell, PopulationAnalyzer

# Set plotting style
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 8)

print("✓ Imports loaded successfully!")

## 1. Helper Functions - CORRECTED

In [ ]:
def get_firing_rates(cell, align_event, window, trial_type=None, direction=None, success_only=True):
    """
    Extract firing rates for specific conditions.
    
    Parameters:
    -----------
    cell : Cell
        Cell object
    align_event : str
        'go_cue' or 'stop_cue'
    window : list
        Time window [start, end] in ms
    trial_type : str, optional
        'GO', 'STOP', or 'CONT'
    direction : int, optional
        0 (right) or 180 (left)
    success_only : bool
        Include only successful trials
    
    Returns:
    --------
    np.array : Firing rates in spikes/sec
    """
    df = cell.filter_trials(trial_type=trial_type, direction=direction, success_only=success_only)
    
    rates = []
    for _, row in df.iterrows():
        if align_event == 'go_cue':
            t0 = row['go_cue']
        elif align_event == 'stop_cue':
            t0 = row['stop_cue']
        else:
            raise ValueError(f"Unknown align_event: {align_event}")
        
        if pd.isna(t0):
            continue
            
        spikes = np.array(row['neural_data'], dtype=float)
        aligned_spikes = spikes - t0
        
        count = np.sum((aligned_spikes >= window[0]) & (aligned_spikes <= window[1]))
        duration = (window[1] - window[0]) / 1000.0
        rates.append(count / duration)
        
    return np.array(rates)


def cohens_d(group1, group2):
    """Calculate Cohen's d effect size."""
    n1, n2 = len(group1), len(group2)
    if n1 < 2 or n2 < 2:
        return np.nan
    var1, var2 = np.var(group1, ddof=1), np.var(group2, ddof=1)
    pooled_std = np.sqrt(((n1 - 1) * var1 + (n2 - 1) * var2) / (n1 + n2 - 2))
    
    if pooled_std == 0:
        return 0.0
    
    return (np.mean(group1) - np.mean(group2)) / pooled_std


def eta_squared(groups):
    """Calculate eta-squared effect size for ANOVA."""
    all_data = np.concatenate(groups)
    grand_mean = np.mean(all_data)
    
    ss_between = sum(len(g) * (np.mean(g) - grand_mean)**2 for g in groups)
    ss_total = np.sum((all_data - grand_mean)**2)
    
    if ss_total == 0:
        return 0.0
    
    return ss_between / ss_total


def direction_selectivity_index(left_rate, right_rate):
    """Calculate direction selectivity index."""
    if left_rate + right_rate == 0:
        return 0.0
    return (left_rate - right_rate) / (left_rate + right_rate)


def modulation_index(baseline_rate, task_rate):
    """Calculate modulation index."""
    if baseline_rate + task_rate == 0:
        return 0.0
    return (task_rate - baseline_rate) / (task_rate + baseline_rate)


def signal_selectivity_index(stop_rate, cont_rate):
    """Calculate signal selectivity index (STOP vs CONT)."""
    if stop_rate + cont_rate == 0:
        return 0.0
    return (stop_rate - cont_rate) / (stop_rate + cont_rate)


def analyze_cell_tuning(cell, alpha=0.05, verbose=False):
    """
    Comprehensive tuning analysis with CORRECTED alignment for STOP/CONT trials.
    
    Key fix: STOP and CONT trials are aligned to stop_cue, not go_cue.
    
    Parameters:
    -----------
    cell : Cell
        Cell object to analyze
    alpha : float
        Significance level (default: 0.05)
    verbose : bool
        Print detailed results
    
    Returns:
    --------
    dict : Comprehensive tuning results
    """
    results = {
        'cell_id': cell.cell_id,
        'cell_type': cell.cell_type,
    }
    
    n_tests = 6  # Increased: task_mod, direction, stop_mod, cont_mod, signal_discrimination, go_vs_signal
    bonferroni_alpha = alpha / n_tests
    
    # ========================================
    # 1. BASELINE (from GO trials)
    # ========================================
    baseline_rates = get_firing_rates(cell, 'go_cue', [-500, 0], trial_type='GO')
    
    # ========================================
    # 2. TASK MODULATION (Baseline vs GO)
    # ========================================
    go_rates = get_firing_rates(cell, 'go_cue', [0, 200], trial_type='GO')
    
    if len(baseline_rates) > 0 and len(go_rates) > 0:
        f_stat, p_val = f_oneway(baseline_rates, go_rates)
        effect_size = cohens_d(go_rates, baseline_rates)
        eta2 = eta_squared([baseline_rates, go_rates])
        
        results['task_modulation'] = {
            'F': f_stat,
            'p': p_val,
            'p_bonferroni': p_val * n_tests,
            'significant': p_val < bonferroni_alpha,
            'cohens_d': effect_size,
            'eta_squared': eta2,
            'modulation_index': modulation_index(np.mean(baseline_rates), np.mean(go_rates)),
            'baseline_mean': np.mean(baseline_rates),
            'go_mean': np.mean(go_rates)
        }
    else:
        results['task_modulation'] = {'significant': False}
    
    # ========================================
    # 3. DIRECTION TUNING (on GO trials)
    # ========================================
    go_left = get_firing_rates(cell, 'go_cue', [0, 200], trial_type='GO', direction=180)
    go_right = get_firing_rates(cell, 'go_cue', [0, 200], trial_type='GO', direction=0)
    
    if len(go_left) > 0 and len(go_right) > 0:
        f_stat, p_val = f_oneway(go_left, go_right)
        effect_size = cohens_d(go_left, go_right)
        dsi = direction_selectivity_index(np.mean(go_left), np.mean(go_right))
        
        results['direction_tuning'] = {
            'F': f_stat,
            'p': p_val,
            'p_bonferroni': p_val * n_tests,
            'significant': p_val < bonferroni_alpha,
            'cohens_d': effect_size,
            'direction_selectivity_index': dsi,
            'preferred_direction': 'left' if dsi > 0 else 'right',
            'left_mean': np.mean(go_left),
            'right_mean': np.mean(go_right)
        }
    else:
        results['direction_tuning'] = {'significant': False}
    
    # ========================================
    # 4. STOP MODULATION (Baseline vs STOP response)
    # CORRECTED: Align to stop_cue!
    # ========================================
    stop_rates = get_firing_rates(cell, 'stop_cue', [0, 200], trial_type='STOP')
    
    if len(baseline_rates) > 0 and len(stop_rates) > 0:
        f_stat, p_val = f_oneway(baseline_rates, stop_rates)
        effect_size = cohens_d(stop_rates, baseline_rates)
        mi = modulation_index(np.mean(baseline_rates), np.mean(stop_rates))
        
        results['stop_modulation'] = {
            'F': f_stat,
            'p': p_val,
            'p_bonferroni': p_val * n_tests,
            'significant': p_val < bonferroni_alpha,
            'cohens_d': effect_size,
            'modulation_index': mi,
            'stop_mean': np.mean(stop_rates)
        }
    else:
        results['stop_modulation'] = {'significant': False}
    
    # ========================================
    # 5. CONT MODULATION (Baseline vs CONT response)
    # CORRECTED: Align to stop_cue!
    # ========================================
    cont_rates = get_firing_rates(cell, 'stop_cue', [0, 200], trial_type='CONT')
    
    if len(baseline_rates) > 0 and len(cont_rates) > 0:
        f_stat, p_val = f_oneway(baseline_rates, cont_rates)
        effect_size = cohens_d(cont_rates, baseline_rates)
        mi = modulation_index(np.mean(baseline_rates), np.mean(cont_rates))
        
        results['cont_modulation'] = {
            'F': f_stat,
            'p': p_val,
            'p_bonferroni': p_val * n_tests,
            'significant': p_val < bonferroni_alpha,
            'cohens_d': effect_size,
            'modulation_index': mi,
            'cont_mean': np.mean(cont_rates)
        }
    else:
        results['cont_modulation'] = {'significant': False}
    
    # ========================================
    # 6. SIGNAL DISCRIMINATION (STOP vs CONT)
    # Both aligned to stop_cue
    # ========================================
    if len(stop_rates) > 0 and len(cont_rates) > 0:
        f_stat, p_val = f_oneway(stop_rates, cont_rates)
        effect_size = cohens_d(stop_rates, cont_rates)
        ssi = signal_selectivity_index(np.mean(stop_rates), np.mean(cont_rates))
        
        results['signal_discrimination'] = {
            'F': f_stat,
            'p': p_val,
            'p_bonferroni': p_val * n_tests,
            'significant': p_val < bonferroni_alpha,
            'cohens_d': effect_size,
            'signal_selectivity_index': ssi,
            'preferred_signal': 'STOP' if ssi > 0 else 'CONT'
        }
    else:
        results['signal_discrimination'] = {'significant': False}
    
    # ========================================
    # 7. GO vs SIGNALS (does signal differ from GO?)
    # ========================================
    if len(go_rates) > 0 and len(stop_rates) > 0 and len(cont_rates) > 0:
        f_stat_stop, p_val_stop = f_oneway(go_rates, stop_rates)
        d_stop = cohens_d(go_rates, stop_rates)
        
        f_stat_cont, p_val_cont = f_oneway(go_rates, cont_rates)
        d_cont = cohens_d(go_rates, cont_rates)
        
        f_stat_all, p_val_all = f_oneway(go_rates, stop_rates, cont_rates)
        
        results['go_vs_signals'] = {
            'F_overall': f_stat_all,
            'p_overall': p_val_all,
            'p_bonferroni': p_val_all * n_tests,
            'significant': p_val_all < bonferroni_alpha,
            'go_vs_stop_p': p_val_stop,
            'go_vs_stop_d': d_stop,
            'go_vs_cont_p': p_val_cont,
            'go_vs_cont_d': d_cont
        }
    else:
        results['go_vs_signals'] = {'significant': False}
    
    # ========================================
    # SUMMARY CLASSIFICATION
    # ========================================
    results['summary'] = {
        'is_task_modulated': results['task_modulation'].get('significant', False),
        'is_direction_tuned': results['direction_tuning'].get('significant', False),
        'is_stop_modulated': results['stop_modulation'].get('significant', False),
        'is_cont_modulated': results['cont_modulation'].get('significant', False),
        'is_signal_discriminative': results['signal_discrimination'].get('significant', False),
        'is_go_vs_signal_different': results['go_vs_signals'].get('significant', False),
    }
    
    # Build tuning profile
    profile = []
    if results['summary']['is_task_modulated']:
        profile.append('task_modulated')
    if results['summary']['is_direction_tuned']:
        profile.append(f"direction_tuned_{results['direction_tuning'].get('preferred_direction', 'none')}")
    if results['summary']['is_stop_modulated']:
        profile.append('stop_modulated')
    if results['summary']['is_cont_modulated']:
        profile.append('cont_modulated')
    if results['summary']['is_signal_discriminative']:
        profile.append(f"signal_selective_{results['signal_discrimination'].get('preferred_signal', 'none')}")
    if results['summary']['is_go_vs_signal_different']:
        profile.append('go_vs_signal_different')
    
    results['summary']['tuning_profile'] = profile if profile else ['not_tuned']
    results['summary']['preferred_direction'] = results['direction_tuning'].get('preferred_direction', 'none')
    
    return results

print("✓ Helper functions defined (CORRECTED)")

## 2. Load Data

In [ ]:
# Load MSN cell database
monkey = 'fiona'
base_path = Path.cwd().parent / 'data' / 'unified_cell_trial_data' 
pickle_file = base_path / f'msn_{monkey}_cell_trial_data.pkl'
cell_df = pd.read_pickle(pickle_file)

print(f"Loaded {len(cell_df):,} trials")
print(f"\nSessions: {cell_df['trial_session'].nunique()}")
print(f"Unique cells: {cell_df['cell_ID'].nunique()}")

## 3. Session Overview

In [ ]:
# Get session statistics
session_stats = []

for session_id in cell_df['trial_session'].unique():
    session_data = cell_df[cell_df['trial_session'] == session_id]
    
    session_stats.append({
        'session_id': session_id,
        'n_cells': session_data['cell_ID'].nunique(),
        'n_trials': len(session_data),
        'n_go': len(session_data[session_data['type'] == 'GO']),
        'n_stop': len(session_data[session_data['type'] == 'STOP']),
        'n_cont': len(session_data[session_data['type'] == 'CONT']),
    })

session_stats_df = pd.DataFrame(session_stats).sort_values('n_cells', ascending=False)

print("\nSession Statistics (sorted by number of cells):")
print("="*80)
display(session_stats_df)

print(f"\nTotal sessions: {len(session_stats_df)}")
print(f"Mean cells per session: {session_stats_df['n_cells'].mean():.1f}")
print(f"Median cells per session: {session_stats_df['n_cells'].median():.1f}")
print(f"Range: {session_stats_df['n_cells'].min()} - {session_stats_df['n_cells'].max()} cells")

## 4. Analyze All Sessions

In [ ]:
# Analyze all cells across all sessions
all_results = []
session_ids = cell_df['trial_session'].unique()

print(f"Analyzing {len(session_ids)} sessions...\n")

for session_id in tqdm(session_ids, desc="Sessions"):
    session_data = cell_df[cell_df['trial_session'] == session_id]
    session = Session(session_data, verbose=False)
    
    for cell_id in session.cell_ids:
        cell = session.get_cell(cell_id)
        result = analyze_cell_tuning(cell, verbose=False)
        result['session_id'] = session_id
        all_results.append(result)

print(f"\n✓ Analysis complete: {len(all_results)} cells analyzed")

## 5. Create Summary DataFrames - CORRECTED

In [ ]:
# Create comprehensive summary DataFrame
summary_data = []

for res in all_results:
    summary_data.append({
        'session_id': res['session_id'],
        'cell_id': res['cell_id'],
        'cell_type': res['cell_type'],
        'is_task_modulated': res['summary']['is_task_modulated'],
        'is_direction_tuned': res['summary']['is_direction_tuned'],
        'is_stop_modulated': res['summary']['is_stop_modulated'],
        'is_cont_modulated': res['summary']['is_cont_modulated'],
        'is_signal_discriminative': res['summary']['is_signal_discriminative'],
        'is_go_vs_signal_different': res['summary']['is_go_vs_signal_different'],
        'preferred_direction': res['summary']['preferred_direction'],
        'preferred_signal': res['signal_discrimination'].get('preferred_signal', 'none'),
        'tuning_profile': ', '.join(res['summary']['tuning_profile']),
        'task_mod_p': res['task_modulation'].get('p', np.nan),
        'dir_tuning_p': res['direction_tuning'].get('p', np.nan),
        'stop_mod_p': res['stop_modulation'].get('p', np.nan),
        'cont_mod_p': res['cont_modulation'].get('p', np.nan),
        'signal_disc_p': res['signal_discrimination'].get('p', np.nan),
        'go_vs_signal_p': res['go_vs_signals'].get('p_overall', np.nan),
        'baseline_FR': res['task_modulation'].get('baseline_mean', np.nan),
        'go_FR': res['task_modulation'].get('go_mean', np.nan),
        'stop_FR': res['stop_modulation'].get('stop_mean', np.nan),
        'cont_FR': res['cont_modulation'].get('cont_mean', np.nan),
        'DSI': res['direction_tuning'].get('direction_selectivity_index', np.nan),
        'SSI': res['signal_discrimination'].get('signal_selectivity_index', np.nan),
        'task_MI': res['task_modulation'].get('modulation_index', np.nan),
        'cohens_d_direction': res['direction_tuning'].get('cohens_d', np.nan),
        'cohens_d_task': res['task_modulation'].get('cohens_d', np.nan),
    })

summary_df = pd.DataFrame(summary_data)

print("Summary DataFrame created (CORRECTED)")
print(f"Shape: {summary_df.shape}")
display(summary_df.head(10))

## 6. Overall Population Statistics - CORRECTED

In [ ]:
print("\n" + "="*80)
print("OVERALL POPULATION STATISTICS (CORRECTED ANALYSIS)")
print("="*80)

total_cells = len(summary_df)
print(f"\nTotal cells analyzed: {total_cells}")
print(f"Total sessions: {summary_df['session_id'].nunique()}")

print(f"\n{'='*80}")
print("TUNING PROPERTIES (across all sessions):")
print(f"{'='*80}")

task_mod = summary_df['is_task_modulated'].sum()
dir_tuned = summary_df['is_direction_tuned'].sum()
stop_mod = summary_df['is_stop_modulated'].sum()
cont_mod = summary_df['is_cont_modulated'].sum()
signal_disc = summary_df['is_signal_discriminative'].sum()
go_vs_sig = summary_df['is_go_vs_signal_different'].sum()

print(f"  Task modulated (GO):       {task_mod:4d} / {total_cells} ({task_mod/total_cells*100:5.1f}%)")
print(f"  Direction tuned:           {dir_tuned:4d} / {total_cells} ({dir_tuned/total_cells*100:5.1f}%)")
print(f"  STOP modulated:            {stop_mod:4d} / {total_cells} ({stop_mod/total_cells*100:5.1f}%)")
print(f"  CONT modulated:            {cont_mod:4d} / {total_cells} ({cont_mod/total_cells*100:5.1f}%)")
print(f"  Signal discriminative:     {signal_disc:4d} / {total_cells} ({signal_disc/total_cells*100:5.1f}%)")
print(f"  GO vs Signal different:    {go_vs_sig:4d} / {total_cells} ({go_vs_sig/total_cells*100:5.1f}%)")

print(f"\n{'='*80}")
print("DIRECTION PREFERENCES (among direction-tuned cells):")
print(f"{'='*80}")

dir_tuned_cells = summary_df[summary_df['is_direction_tuned']]
if len(dir_tuned_cells) > 0:
    dir_counts = dir_tuned_cells['preferred_direction'].value_counts()
    for direction, count in dir_counts.items():
        print(f"  {direction.capitalize():10s}: {count:4d} ({count/len(dir_tuned_cells)*100:5.1f}%)")
else:
    print("  No direction-tuned cells found")

print(f"\n{'='*80}")
print("SIGNAL PREFERENCES (among signal-discriminative cells):")
print(f"{'='*80}")

sig_disc_cells = summary_df[summary_df['is_signal_discriminative']]
if len(sig_disc_cells) > 0:
    sig_counts = sig_disc_cells['preferred_signal'].value_counts()
    for signal, count in sig_counts.items():
        print(f"  {signal:10s}: {count:4d} ({count/len(sig_disc_cells)*100:5.1f}%)")
else:
    print("  No signal-discriminative cells found")

print(f"\n{'='*80}")
print("MOST COMMON TUNING PROFILES:")
print(f"{'='*80}")

profile_counts = summary_df['tuning_profile'].value_counts().head(10)
for profile, count in profile_counts.items():
    print(f"  {profile:60s}: {count:4d} ({count/total_cells*100:5.1f}%)")

## 7. Session-Level Statistics - CORRECTED

In [ ]:
# Calculate statistics per session
session_tuning_stats = []

for session_id in summary_df['session_id'].unique():
    session_cells = summary_df[summary_df['session_id'] == session_id]
    n_cells = len(session_cells)
    
    session_tuning_stats.append({
        'session_id': session_id,
        'n_cells': n_cells,
        'n_task_modulated': session_cells['is_task_modulated'].sum(),
        'n_direction_tuned': session_cells['is_direction_tuned'].sum(),
        'n_stop_modulated': session_cells['is_stop_modulated'].sum(),
        'n_cont_modulated': session_cells['is_cont_modulated'].sum(),
        'n_signal_discriminative': session_cells['is_signal_discriminative'].sum(),
        'n_go_vs_signal_different': session_cells['is_go_vs_signal_different'].sum(),
        'pct_task_modulated': session_cells['is_task_modulated'].sum() / n_cells * 100,
        'pct_direction_tuned': session_cells['is_direction_tuned'].sum() / n_cells * 100,
        'pct_stop_modulated': session_cells['is_stop_modulated'].sum() / n_cells * 100,
        'pct_cont_modulated': session_cells['is_cont_modulated'].sum() / n_cells * 100,
        'pct_signal_discriminative': session_cells['is_signal_discriminative'].sum() / n_cells * 100,
        'pct_go_vs_signal_different': session_cells['is_go_vs_signal_different'].sum() / n_cells * 100,
        'mean_baseline_FR': session_cells['baseline_FR'].mean(),
        'mean_go_FR': session_cells['go_FR'].mean(),
        'mean_stop_FR': session_cells['stop_FR'].mean(),
        'mean_cont_FR': session_cells['cont_FR'].mean(),
        'mean_DSI': session_cells['DSI'].abs().mean(),
        'mean_SSI': session_cells['SSI'].abs().mean(),
        'mean_task_MI': session_cells['task_MI'].abs().mean(),
    })

session_tuning_df = pd.DataFrame(session_tuning_stats).sort_values('n_cells', ascending=False)

print("\nSession-Level Tuning Statistics (CORRECTED):")
print("="*80)
display(session_tuning_df)

## 8. Visualizations

### 8.1 Overall Distribution of Tuning Properties

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(18, 12))

# 1. Tuning category counts
ax = axes[0, 0]
tuning_counts = pd.Series({
    'Task\nModulated': summary_df['is_task_modulated'].sum(),
    'Direction\nTuned': summary_df['is_direction_tuned'].sum(),
    'STOP\nModulated': summary_df['is_stop_modulated'].sum(),
    'CONT\nModulated': summary_df['is_cont_modulated'].sum(),
    'Signal\nDiscriminative': summary_df['is_signal_discriminative'].sum(),
    'GO vs\nSignal Diff': summary_df['is_go_vs_signal_different'].sum()
})
bars = ax.bar(range(len(tuning_counts)), tuning_counts.values, 
              color=['#1f77b4', '#ff7f0e', '#2ca02c', '#d62728', '#9467bd', '#8c564b'], alpha=0.7)
ax.set_xticks(range(len(tuning_counts)))
ax.set_xticklabels(tuning_counts.index, fontsize=9)
ax.set_ylabel('Number of Cells')
ax.set_title(f'Tuning Categories (N={total_cells} cells) - CORRECTED')
ax.axhline(total_cells/2, color='gray', linestyle='--', alpha=0.5)
for bar, count in zip(bars, tuning_counts.values):
    height = bar.get_height()
    ax.text(bar.get_x() + bar.get_width()/2., height,
            f'{int(count)}\n({count/total_cells*100:.1f}%)',
            ha='center', va='bottom', fontsize=8)

# 2. Direction Selectivity Index
ax = axes[0, 1]
dsi_values = summary_df['DSI'].dropna()
ax.hist(dsi_values, bins=40, edgecolor='black', alpha=0.7)
ax.axvline(0, color='red', linestyle='--', linewidth=2)
ax.set_xlabel('Direction Selectivity Index (DSI)')
ax.set_ylabel('Count')
ax.set_title(f'Direction Selectivity (n={len(dsi_values)})')
ax.text(0.02, 0.98, f'Left-preferring: {(dsi_values > 0).sum()}\nRight-preferring: {(dsi_values < 0).sum()}',
        transform=ax.transAxes, verticalalignment='top',
        bbox=dict(boxstyle='round', facecolor='white', alpha=0.8))

# 3. Signal Selectivity Index
ax = axes[0, 2]
ssi_values = summary_df['SSI'].dropna()
ax.hist(ssi_values, bins=40, edgecolor='black', alpha=0.7, color='purple')
ax.axvline(0, color='red', linestyle='--', linewidth=2)
ax.set_xlabel('Signal Selectivity Index (SSI)')
ax.set_ylabel('Count')
ax.set_title(f'Signal Selectivity (n={len(ssi_values)}) - NEW')
ax.text(0.02, 0.98, f'STOP-preferring: {(ssi_values > 0).sum()}\nCONT-preferring: {(ssi_values < 0).sum()}',
        transform=ax.transAxes, verticalalignment='top',
        bbox=dict(boxstyle='round', facecolor='white', alpha=0.8))

# 4. Baseline vs GO Firing Rates
ax = axes[1, 0]
baseline_valid = summary_df['baseline_FR'].dropna()
go_valid = summary_df.loc[baseline_valid.index, 'go_FR'].dropna()
baseline_valid = summary_df.loc[go_valid.index, 'baseline_FR']
ax.scatter(baseline_valid, go_valid, alpha=0.3, s=20)
max_val = max(baseline_valid.max(), go_valid.max())
ax.plot([0, max_val], [0, max_val], 'r--', linewidth=2, label='Unity')
ax.set_xlabel('Baseline FR (sp/s)')
ax.set_ylabel('GO FR (sp/s)')
ax.set_title('Baseline vs GO Firing Rates')
ax.legend()
ax.set_xlim(0, None)
ax.set_ylim(0, None)

# 5. STOP vs CONT Firing Rates
ax = axes[1, 1]
stop_valid = summary_df['stop_FR'].dropna()
cont_valid = summary_df.loc[stop_valid.index, 'cont_FR'].dropna()
stop_valid = summary_df.loc[cont_valid.index, 'stop_FR']
ax.scatter(stop_valid, cont_valid, alpha=0.3, s=20, color='purple')
max_val = max(stop_valid.max(), cont_valid.max())
ax.plot([0, max_val], [0, max_val], 'r--', linewidth=2, label='Unity')
ax.set_xlabel('STOP FR (sp/s)')
ax.set_ylabel('CONT FR (sp/s)')
ax.set_title('STOP vs CONT Firing Rates - NEW')
ax.legend()
ax.set_xlim(0, None)
ax.set_ylim(0, None)

# 6. P-value distributions
ax = axes[1, 2]
p_cols = ['task_mod_p', 'dir_tuning_p', 'stop_mod_p', 'cont_mod_p', 'signal_disc_p', 'go_vs_signal_p']
labels = ['Task Mod', 'Direction', 'STOP Mod', 'CONT Mod', 'Signal Disc', 'GO vs Sig']
colors = ['#1f77b4', '#ff7f0e', '#2ca02c', '#d62728', '#9467bd', '#8c564b']
for col, label, color in zip(p_cols, labels, colors):
    ax.hist(summary_df[col].dropna(), bins=20, alpha=0.4, label=label, color=color)
ax.axvline(0.05, color='red', linestyle='--', linewidth=2, label='α=0.05')
ax.axvline(0.01, color='darkred', linestyle='-', linewidth=2, label='α=0.01')
ax.set_xlabel('P-value')
ax.set_ylabel('Count')
ax.set_title('P-value Distributions - CORRECTED')
ax.legend(fontsize=8)
ax.set_xlim(0, 1)

plt.tight_layout()
plt.show()

### 8.2 Session-Level Comparisons

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(16, 12))

# 1. Percentage of task-modulated cells per session
ax = axes[0, 0]
session_tuning_df_sorted = session_tuning_df.sort_values('pct_task_modulated')
ax.barh(range(len(session_tuning_df_sorted)), session_tuning_df_sorted['pct_task_modulated'],
        color='steelblue', alpha=0.7)
ax.set_yticks(range(len(session_tuning_df_sorted)))
ax.set_yticklabels(session_tuning_df_sorted['session_id'], fontsize=8)
ax.set_xlabel('% Task Modulated')
ax.set_title('Task Modulation by Session')
ax.axvline(session_tuning_df['pct_task_modulated'].mean(), 
           color='red', linestyle='--', linewidth=2, label='Mean')
ax.legend()

# 2. Percentage of signal-discriminative cells per session
ax = axes[0, 1]
session_tuning_df_sorted = session_tuning_df.sort_values('pct_signal_discriminative')
ax.barh(range(len(session_tuning_df_sorted)), session_tuning_df_sorted['pct_signal_discriminative'],
        color='purple', alpha=0.7)
ax.set_yticks(range(len(session_tuning_df_sorted)))
ax.set_yticklabels(session_tuning_df_sorted['session_id'], fontsize=8)
ax.set_xlabel('% Signal Discriminative')
ax.set_title('Signal Discrimination by Session - NEW')
ax.axvline(session_tuning_df['pct_signal_discriminative'].mean(), 
           color='red', linestyle='--', linewidth=2, label='Mean')
ax.legend()

# 3. Number of cells vs percentage tuned
ax = axes[1, 0]
ax.scatter(session_tuning_df['n_cells'], session_tuning_df['pct_task_modulated'], 
          s=100, alpha=0.6, label='Task Mod')
ax.scatter(session_tuning_df['n_cells'], session_tuning_df['pct_direction_tuned'], 
          s=100, alpha=0.6, label='Direction')
ax.scatter(session_tuning_df['n_cells'], session_tuning_df['pct_signal_discriminative'], 
          s=100, alpha=0.6, label='Signal Disc', color='purple')
ax.set_xlabel('Number of Cells in Session')
ax.set_ylabel('% Tuned Cells')
ax.set_title('Session Size vs Tuning Proportion')
ax.legend()
ax.grid(True, alpha=0.3)

# 4. Heatmap of tuning properties across sessions - CORRECTED
ax = axes[1, 1]
heatmap_data = session_tuning_df.set_index('session_id')[[
    'pct_task_modulated', 'pct_direction_tuned', 
    'pct_stop_modulated', 'pct_cont_modulated',
    'pct_signal_discriminative', 'pct_go_vs_signal_different'
]].T
im = ax.imshow(heatmap_data.values, cmap='YlOrRd', aspect='auto')
ax.set_yticks(range(len(heatmap_data.index)))
ax.set_yticklabels(['Task Mod', 'Direction', 'STOP Mod', 'CONT Mod', 'Signal Disc', 'GO vs Sig'], fontsize=10)
ax.set_xticks(range(len(heatmap_data.columns)))
ax.set_xticklabels(heatmap_data.columns, rotation=90, fontsize=8)
ax.set_title('Tuning Properties Heatmap - CORRECTED')
plt.colorbar(im, ax=ax, label='% of cells')

plt.tight_layout()
plt.show()

### 8.3 Sessions with Highest/Lowest Tuning

In [ ]:
print("\nSessions with HIGHEST task modulation:")
print("="*80)
display(session_tuning_df.nlargest(5, 'pct_task_modulated')[[
    'session_id', 'n_cells', 'n_task_modulated', 'pct_task_modulated'
]])

print("\nSessions with HIGHEST direction tuning:")
print("="*80)
display(session_tuning_df.nlargest(5, 'pct_direction_tuned')[[
    'session_id', 'n_cells', 'n_direction_tuned', 'pct_direction_tuned'
]])

print("\nSessions with HIGHEST signal discrimination - NEW:")
print("="*80)
display(session_tuning_df.nlargest(5, 'pct_signal_discriminative')[[
    'session_id', 'n_cells', 'n_signal_discriminative', 'pct_signal_discriminative'
]])

print("\nSessions with MOST cells:")
print("="*80)
display(session_tuning_df.nlargest(5, 'n_cells')[[
    'session_id', 'n_cells', 'pct_task_modulated', 'pct_direction_tuned', 'pct_signal_discriminative'
]])

## 9. Save Results - CORRECTED

In [ ]:
# Create output directory
output_path = Path.cwd().parent / 'data' / 'tuning_analysis_v2'
output_path.mkdir(exist_ok=True)

# Save overall summary
csv_file = output_path / f'tuning_summary_all_sessions_{monkey}.csv'
summary_df.to_csv(csv_file, index=False)
print(f"✓ Saved overall summary to: {csv_file}")

# Save session-level statistics
session_csv = output_path / f'session_tuning_stats_{monkey}.csv'
session_tuning_df.to_csv(session_csv, index=False)
print(f"✓ Saved session statistics to: {session_csv}")

# Save full results as pickle
pkl_file = output_path / f'tuning_full_results_all_sessions_{monkey}.pkl'
pd.to_pickle(all_results, pkl_file)
print(f"✓ Saved full results to: {pkl_file}")

# Save individual session files
print(f"\nSaving individual session files...")
for session_id in tqdm(summary_df['session_id'].unique(), desc="Saving sessions"):
    session_data = summary_df[summary_df['session_id'] == session_id]
    session_file = output_path / f'tuning_summary_{session_id}.csv'
    session_data.to_csv(session_file, index=False)

print(f"\n✓ All results saved to: {output_path}")
print(f"\nNOTE: Results saved with CORRECTED alignment (STOP/CONT to stop_cue)")

## 10. Statistical Tests Across Sessions

In [ ]:
print("\n" + "="*80)
print("STATISTICAL TESTS - CORRECTED")
print("="*80)

# Test: Do sessions differ in proportion of task-modulated cells?
print("\n1. Variability in task modulation across sessions:")
print("-"*80)
print(f"Mean: {session_tuning_df['pct_task_modulated'].mean():.1f}%")
print(f"Std:  {session_tuning_df['pct_task_modulated'].std():.1f}%")
print(f"Range: {session_tuning_df['pct_task_modulated'].min():.1f}% - {session_tuning_df['pct_task_modulated'].max():.1f}%")

# Test: Do sessions differ in proportion of direction-tuned cells?
print("\n2. Variability in direction tuning across sessions:")
print("-"*80)
print(f"Mean: {session_tuning_df['pct_direction_tuned'].mean():.1f}%")
print(f"Std:  {session_tuning_df['pct_direction_tuned'].std():.1f}%")
print(f"Range: {session_tuning_df['pct_direction_tuned'].min():.1f}% - {session_tuning_df['pct_direction_tuned'].max():.1f}%")

# Test: Do sessions differ in proportion of signal-discriminative cells? (NEW)
print("\n3. Variability in signal discrimination across sessions - NEW:")
print("-"*80)
print(f"Mean: {session_tuning_df['pct_signal_discriminative'].mean():.1f}%")
print(f"Std:  {session_tuning_df['pct_signal_discriminative'].std():.1f}%")
print(f"Range: {session_tuning_df['pct_signal_discriminative'].min():.1f}% - {session_tuning_df['pct_signal_discriminative'].max():.1f}%")

# Test: Correlation between session size and tuning proportion
from scipy.stats import pearsonr, spearmanr

print("\n4. Correlation: Session size vs tuning proportions:")
print("-"*80)

r_task, p_task = pearsonr(session_tuning_df['n_cells'], session_tuning_df['pct_task_modulated'])
print(f"Task modulation: r={r_task:.3f}, p={p_task:.4f}")

r_dir, p_dir = pearsonr(session_tuning_df['n_cells'], session_tuning_df['pct_direction_tuned'])
print(f"Direction tuning: r={r_dir:.3f}, p={p_dir:.4f}")

r_sig, p_sig = pearsonr(session_tuning_df['n_cells'], session_tuning_df['pct_signal_discriminative'])
print(f"Signal discrimination: r={r_sig:.3f}, p={p_sig:.4f}")

# Test: Are mean firing rates different between tuned and non-tuned cells?
print("\n5. Firing rates: Tuned vs Non-tuned cells:")
print("-"*80)

task_mod_FR = summary_df[summary_df['is_task_modulated']]['baseline_FR'].dropna()
not_task_mod_FR = summary_df[~summary_df['is_task_modulated']]['baseline_FR'].dropna()

if len(task_mod_FR) > 0 and len(not_task_mod_FR) > 0:
    t_stat, p_val = stats.ttest_ind(task_mod_FR, not_task_mod_FR)
    print(f"Task-modulated: {task_mod_FR.mean():.2f} ± {task_mod_FR.std():.2f} sp/s")
    print(f"Not modulated:  {not_task_mod_FR.mean():.2f} ± {not_task_mod_FR.std():.2f} sp/s")
    print(f"t-test: t={t_stat:.3f}, p={p_val:.4f}")

dir_tuned_FR = summary_df[summary_df['is_direction_tuned']]['baseline_FR'].dropna()
not_dir_tuned_FR = summary_df[~summary_df['is_direction_tuned']]['baseline_FR'].dropna()

if len(dir_tuned_FR) > 0 and len(not_dir_tuned_FR) > 0:
    t_stat, p_val = stats.ttest_ind(dir_tuned_FR, not_dir_tuned_FR)
    print(f"\nDirection-tuned: {dir_tuned_FR.mean():.2f} ± {dir_tuned_FR.std():.2f} sp/s")
    print(f"Not tuned:       {not_dir_tuned_FR.mean():.2f} ± {not_dir_tuned_FR.std():.2f} sp/s")
    print(f"t-test: t={t_stat:.3f}, p={p_val:.4f}")

sig_disc_FR = summary_df[summary_df['is_signal_discriminative']]['baseline_FR'].dropna()
not_sig_disc_FR = summary_df[~summary_df['is_signal_discriminative']]['baseline_FR'].dropna()

if len(sig_disc_FR) > 0 and len(not_sig_disc_FR) > 0:
    t_stat, p_val = stats.ttest_ind(sig_disc_FR, not_sig_disc_FR)
    print(f"\nSignal-discriminative: {sig_disc_FR.mean():.2f} ± {sig_disc_FR.std():.2f} sp/s")
    print(f"Not discriminative:    {not_sig_disc_FR.mean():.2f} ± {not_sig_disc_FR.std():.2f} sp/s")
    print(f"t-test: t={t_stat:.3f}, p={p_val:.4f}")

## 11. Summary Report

In [ ]:
# Generate summary report
report = f"""
{'='*80}
MULTI-SESSION TUNING ANALYSIS SUMMARY REPORT - CORRECTED
{'='*80}

Dataset: {monkey.capitalize()}
Date: {pd.Timestamp.now().strftime('%Y-%m-%d %H:%M:%S')}

IMPORTANT: This analysis uses CORRECTED alignment:
- STOP/CONT trials aligned to stop_cue (not go_cue)
- Measures actual signal responses, not initial GO responses

{'='*80}
DATASET OVERVIEW
{'='*80}
Total sessions analyzed: {summary_df['session_id'].nunique()}
Total cells analyzed: {len(summary_df)}
Mean cells per session: {session_tuning_df['n_cells'].mean():.1f} (range: {session_tuning_df['n_cells'].min()}-{session_tuning_df['n_cells'].max()})

{'='*80}
OVERALL TUNING STATISTICS (CORRECTED)
{'='*80}
Task modulated (GO):       {summary_df['is_task_modulated'].sum():4d} / {len(summary_df)} ({summary_df['is_task_modulated'].sum()/len(summary_df)*100:5.1f}%)
Direction tuned:           {summary_df['is_direction_tuned'].sum():4d} / {len(summary_df)} ({summary_df['is_direction_tuned'].sum()/len(summary_df)*100:5.1f}%)
STOP modulated:            {summary_df['is_stop_modulated'].sum():4d} / {len(summary_df)} ({summary_df['is_stop_modulated'].sum()/len(summary_df)*100:5.1f}%)
CONT modulated:            {summary_df['is_cont_modulated'].sum():4d} / {len(summary_df)} ({summary_df['is_cont_modulated'].sum()/len(summary_df)*100:5.1f}%)
Signal discriminative:     {summary_df['is_signal_discriminative'].sum():4d} / {len(summary_df)} ({summary_df['is_signal_discriminative'].sum()/len(summary_df)*100:5.1f}%)
GO vs Signal different:    {summary_df['is_go_vs_signal_different'].sum():4d} / {len(summary_df)} ({summary_df['is_go_vs_signal_different'].sum()/len(summary_df)*100:5.1f}%)

{'='*80}
DIRECTION PREFERENCES (among direction-tuned cells)
{'='*80}
"""

dir_tuned_cells = summary_df[summary_df['is_direction_tuned']]
if len(dir_tuned_cells) > 0:
    dir_counts = dir_tuned_cells['preferred_direction'].value_counts()
    for direction, count in dir_counts.items():
        report += f"{direction.capitalize():10s}: {count:4d} ({count/len(dir_tuned_cells)*100:5.1f}%)\n"
else:
    report += "No direction-tuned cells found\n"

report += f"""
{'='*80}
SIGNAL PREFERENCES (among signal-discriminative cells) - NEW
{'='*80}
"""

sig_disc_cells = summary_df[summary_df['is_signal_discriminative']]
if len(sig_disc_cells) > 0:
    sig_counts = sig_disc_cells['preferred_signal'].value_counts()
    for signal, count in sig_counts.items():
        report += f"{signal:10s}: {count:4d} ({count/len(sig_disc_cells)*100:5.1f}%)\n"
else:
    report += "No signal-discriminative cells found\n"

report += f"""
{'='*80}
SESSION-LEVEL VARIABILITY
{'='*80}
Task modulation:       {session_tuning_df['pct_task_modulated'].mean():.1f}% ± {session_tuning_df['pct_task_modulated'].std():.1f}%
Direction tuning:      {session_tuning_df['pct_direction_tuned'].mean():.1f}% ± {session_tuning_df['pct_direction_tuned'].std():.1f}%
Signal discrimination: {session_tuning_df['pct_signal_discriminative'].mean():.1f}% ± {session_tuning_df['pct_signal_discriminative'].std():.1f}%

{'='*80}
TOP SESSIONS (by number of cells)
{'='*80}
"""

top_sessions = session_tuning_df.nlargest(5, 'n_cells')
for _, row in top_sessions.iterrows():
    report += f"{row['session_id']:15s}: {int(row['n_cells']):3d} cells, "
    report += f"Task: {row['pct_task_modulated']:5.1f}%, Dir: {row['pct_direction_tuned']:5.1f}%, Sig: {row['pct_signal_discriminative']:5.1f}%\n"

report += f"""
{'='*80}
KEY CORRECTIONS IN THIS ANALYSIS
{'='*80}
1. STOP/CONT trials aligned to stop_cue (not go_cue)
2. New measures: STOP modulation, CONT modulation, Signal discrimination
3. Signal Selectivity Index (SSI) added
4. 6 tests instead of 5 (Bonferroni correction adjusted)

{'='*80}
FILES SAVED
{'='*80}
Overall summary:     tuning_summary_all_sessions_{monkey}.csv
Session statistics:  session_tuning_stats_{monkey}.csv
Full results:        tuning_full_results_all_sessions_{monkey}.pkl
Individual sessions: tuning_summary_[session_id].csv (one per session)

Location: {output_path}
{'='*80}
"""

print(report)

# Save report
report_file = output_path / f'tuning_analysis_report_{monkey}.txt'
with open(report_file, 'w') as f:
    f.write(report)
print(f"\n✓ Report saved to: {report_file}")